In [19]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [20]:
import warnings
warnings.simplefilter("ignore")

In [29]:
rating_df = pd.read_csv(r"C:\Users\amann\Aman\Aman\MLOPS\Project-2\Anime_Recommender_System\artifacts\raw\animelist[1].csv")

In [30]:
rating_df

,user_id,anime_id,rating,watching_status,watched_episodes
0,0,67,9,1,1
1,0,6702,7,1,4
2,0,242,10,1,4
3,0,4898,0,1,1
4,0,21,10,1,0
...,...,...,...,...,...
4999995,16508,21405,8,2,12
4999996,16508,24913,9,2,1
4999997,16508,37451,7,2,18
4999998,16508,28755,8,2,1


In [31]:
rating_df = rating_df.drop(columns=["watching_status", "watched_episodes"])

In [32]:
rating_df.shape[0]

5000000

In [33]:
num_ratings = rating_df['user_id'].value_counts()

In [34]:
num_ratings

user_id
11100    14429
10255     8403
4773      5735
6852      5406
16057     5080
         ...  
16363        1
16377        1
16450        1
16465        1
16480        1
Name: count, Length: 15186, dtype: int64

#### We will keep only those users who have watched over 400 anime as we can trust their experience when it comes to rating different kinds of anime

In [35]:
rating_df = rating_df[rating_df['user_id'].isin(num_ratings[num_ratings >= 400].index)].copy()

In [36]:
rating_df.shape[0]

3246641

In [41]:
min_rating = min(rating_df["rating"])

In [42]:
max_rating = max(rating_df["rating"])

In [46]:
rating_df['rating'] = rating_df['rating'].apply(lambda x: (x - min_rating)/(max_rating - min_rating)).values.astype(np.float64)

In [55]:
rating_df.sample(10)

,user_id,anime_id,rating
4481810,14732,38883,0.7
3877737,12837,33184,0.3
1814006,6174,28851,0.8
2598358,8753,6324,0.0
10659,42,32768,0.8
3514602,11650,22961,0.8
869487,2952,30015,0.8
3331420,11100,15959,0.0
2061861,6979,8074,0.6
4232149,13929,31741,0.0


In [49]:
min(rating_df['rating'])

0.0

In [50]:
max(rating_df['rating'])

1.0

In [52]:
np.mean(rating_df['rating'])

np.float64(0.4122732695114736)

In [56]:
rating_df.duplicated().sum()

np.int64(0)

In [57]:
rating_df.isnull().sum()

user_id     0
anime_id    0
rating      0
dtype: int64

In [72]:
user_ids = rating_df['user_id'].unique().tolist()
user2user_encoded = {x : i for i, x in enumerate(user_ids)}
user2user_decoded = {i : x for i, x in enumerate(user_ids)}
rating_df['user'] = rating_df['user_id'].map(user2user_encoded)

In [73]:
rating_df.head()

,user_id,anime_id,rating,user
213,2,24833,0.0,0
214,2,235,1.0,0
215,2,36721,0.0,0
216,2,40956,0.0,0
217,2,31933,0.0,0


In [74]:
len(user_ids)

4203

In [75]:
anime_ids = rating_df['anime_id'].unique().tolist()
anime2anime_encoded = {x : i for i, x in enumerate(anime_ids)}
anime2anime_decoded = {i : x for i, x in enumerate(anime_ids)}
rating_df['anime'] = rating_df['anime_id'].map(anime2anime_encoded)

In [76]:
rating_df

,user_id,anime_id,rating,user,anime
213,2,24833,0.0,0,0
214,2,235,1.0,0,1
215,2,36721,0.0,0,2
216,2,40956,0.0,0,3
217,2,31933,0.0,0,4
...,...,...,...,...,...
4999916,16507,8985,0.0,4202,2533
4999917,16507,5454,0.0,4202,817
4999918,16507,15911,0.0,4202,2455
4999919,16507,878,0.0,4202,2154


In [77]:
len(anime_ids)

17149

In [78]:
rating_df = rating_df.sample(frac=1, random_state=43).reset_index(drop=True)

In [79]:
rating_df

,user_id,anime_id,rating,user,anime
0,457,18153,0.9,120,1377
1,4903,20507,0.7,1195,1216
2,6313,23325,0.0,1591,1239
3,15851,37491,0.0,4024,1813
4,1596,29803,0.9,415,353
...,...,...,...,...,...
3246636,7916,721,1.0,2005,955
3246637,7516,16067,0.0,1898,1398
3246638,12682,28171,0.8,3208,67
3246639,8387,33255,0.7,2114,1595


In [80]:
X = rating_df[["user", "anime"]].values
y = rating_df['rating']

In [82]:
test_size = 1000
train_size = rating_df.shape[0] - test_size

In [84]:
X_train, X_test, y_train, y_test = (
    X[: train_size],
    X[train_size :],
    y[: train_size],
    y[train_size :]
)

In [87]:
type(X_train)

numpy.ndarray

In [88]:
X_train_array = [X_train[:, 0], X_train[:, 1]]
X_test_array = [X_test[:, 0], X_test[:, 1]]

In [89]:
type(X_train_array)

list

In [90]:
type(X_train_array[0])

numpy.ndarray

##### MODEL ARCHITECTURE